# Elder Mist Round 2 → learn RL by actually fighting it

This is a **lightweight mechanics-inspired replica** of the second Elder Mist fight.

No Sea of Stars install is needed. No GPU is needed. The whole battle is just Python state.

The real encounter facts we preserve are:

- only **Zale + Valere** fight the second encounter
- Elder Mist has **3,995 HP**
- his Sword has **800 HP**
- attacking Elder Mist's body while the Sword is active triggers **Counter Slash**
- disabling the Sword suppresses that counter; it returns after a few turns
- the real second-fight Sword returns at **489 HP**
- splash damage aimed at the Sword can damage both targets without provoking Counter Slash
- Torblado / Thunder / Healing Mist are lock-based charged moves

The damage numbers on *our* Zale/Valere side are normalized so that we can run thousands of episodes quickly.

---

## The one loop to keep in your head

```text
state / observation
        ↓
      POLICY
        ↓
      ACTION
        ↓
  ENVIRONMENT.step()
        ↓
new state + REWARD
        ↓
       learn
```

We will literally inspect every one of these pieces.


In [ ]:
# Run this once if the packages are missing.
# In Jupyter, uncomment:
# %pip install -q numpy pandas matplotlib gymnasium ipywidgets

from pathlib import Path
import sys

# Works when the notebook is opened from elder-mist-rl-lab/notebooks/
repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from eldermist import ElderMistReplicaEnv, ACTIONS, ACTION_TO_ID
from eldermist.policies import random_policy, sensible_rule_policy
from eldermist.qlearning import train_q, q_policy, valid_ids
from eldermist.viz import plot_state

print("Loaded. Actions:")
for i, a in enumerate(ACTIONS):
    print(i, a)


## 1 — `reset()` does NOT mean “no state”

`reset()` means:

> create a fresh episode and return the first observation.

The environment immediately contains a complete battle state: boss HP, sword HP, party HP, MP, locks, whose turn it is, etc.


In [ ]:
env = ElderMistReplicaEnv()
obs, info = env.reset(seed=7)

print("obs shape:", obs.shape)
print("first numeric observation:")
print(obs)
print()
print(env.render())

fig = plot_state(env, "Episode begins")
plt.show()


Notice the distinction:

```text
TRUE ENVIRONMENT STATE
boss HP, sword HP, exact lock counts, MP, shields, timers, ...

        ↓ observation function

OBSERVATION
a numeric vector the policy is allowed to receive
```

In this simulator we *can* inspect the true state because we wrote the environment.

In the real world — or LifeOS — you generally cannot.


## 2 — Play it yourself: **you are the policy**

Click actions below.

While you play, watch for the Sword rule:

> If the Sword is active and you directly hit Elder Mist's body, you trigger Counter Slash.

That means the state variable `sword_active` can change which action is good.

This will matter when we later deliberately hide it from an RL agent.


In [ ]:
manual = ElderMistReplicaEnv(reward_mode="shaped")
manual.reset(seed=11)

out = widgets.Output()
buttons = {a: widgets.Button(description=a.replace("_", " "), layout=widgets.Layout(width="180px"))
           for a in ACTIONS}
reset_button = widgets.Button(description="RESET FIGHT", button_style="warning")

def redraw():
    with out:
        clear_output(wait=True)
        fig = plot_state(manual)
        display(fig)
        plt.close(fig)
        print()
        print("Compact state:", manual.compact_state())
        print("Valid actions:", manual.valid_actions())

    valid = set(manual.valid_actions())
    for name, b in buttons.items():
        b.disabled = name not in valid or manual._done()

def click_action(name):
    def _cb(_):
        _, reward, terminated, truncated, info = manual.step(ACTION_TO_ID[name])
        with out:
            print(f"\nreward = {reward:+.3f}")
        redraw()
    return _cb

for name, button in buttons.items():
    button.on_click(click_action(name))

def do_reset(_):
    manual.reset(seed=11)
    redraw()

reset_button.on_click(do_reset)

display(widgets.HTML("<b>YOU are π (the policy). Choose Aₜ:</b>"))
display(widgets.VBox([
    widgets.HBox(list(buttons.values())[:4]),
    widgets.HBox(list(buttons.values())[4:]),
    reset_button,
    out
]))
redraw()


## 3 — Where is the transition function \(T\)?

Suppose you choose:

```text
sunball_sword
```

Your policy only chose the action.

**The environment** decides:

- how much Sword HP falls
- how much splash damage reaches Elder Mist
- which locks break
- whether a charged move fires
- whether the boss attacks
- whether the Sword revives

That code is the transition dynamics:

```text
T: (current state, action) → next state
```

So in a game, **the combat engine is the environment dynamics**.

The policy is not the combat engine.


## 4 — A policy does not have to be a neural network

Here's a readable hand-written policy:

```text
if we're dying and Mending Light is available:
    heal

else if Sword is active:
    prefer splash attack aimed at Sword

else:
    attack Elder Mist's body
```

That's still a policy because it maps a situation → action.


In [ ]:
from eldermist.policies import explain_sensible_rule_policy

demo = ElderMistReplicaEnv()
demo.reset(seed=8)

action_name, reason = explain_sensible_rule_policy(demo)

print("Current represented situation:")
print(demo.compact_state())
print()
print("Rule policy chose:", action_name)
print("WHY:")
print(reason)


The crucial phrase is **rule policy**. This move happened because *we encoded the reasoning explicitly*. Later, when Q-learning chooses a move, we'll inspect learned Q-values instead.

In [ ]:
def run_episode(policy, seed=0, verbose=False):
    e = ElderMistReplicaEnv()
    obs, _ = e.reset(seed=seed)
    total = 0.0
    rows = []

    while True:
        before = e.compact_state()
        action = policy(e, obs)
        obs, reward, terminated, truncated, info = e.step(action)
        total += reward

        rows.append({
            "turn": e.state.turn,
            "state_before": before,
            "action": ACTIONS[action],
            "reward": reward,
            "boss_hp": e.state.boss_hp,
            "sword_active": e.state.sword_active,
            "sword_hp": e.state.sword_hp,
            "zale_hp": e.state.zale_hp,
            "valere_hp": e.state.valere_hp,
            "event": e.last_event,
        })

        if verbose:
            print(rows[-1]["turn"], rows[-1]["action"], "->", e.last_event)

        if terminated or truncated:
            break

    return e, total, pd.DataFrame(rows)

random_end, random_return, random_df = run_episode(random_policy, seed=2)
rule_end, rule_return, rule_df = run_episode(sensible_rule_policy, seed=2)

print("Random policy:")
print("  win =", random_end.state.boss_hp <= 0,
      "turns =", random_end.state.turn,
      "counter slashes =", random_end.state.counter_slashes,
      "return =", round(random_return, 3))

print("\nRule policy:")
print("  win =", rule_end.state.boss_hp <= 0,
      "turns =", rule_end.state.turn,
      "counter slashes =", rule_end.state.counter_slashes,
      "return =", round(rule_return, 3))


The rule policy already knows the Sword trick because **we programmed that knowledge into the policy**.

RL asks a different question:

> Can repeated experience and reward cause a policy to discover useful choices?


## 5 — What does “learning” mean here?

We'll use tabular Q-learning first because you can inspect the learned numbers.

For every state/action pair, store:

\[
Q(s,a)
\]

roughly:

> “How much future reward has this action tended to produce from situations like this?”

The update is:

\[
Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'}Q(s',a') - Q(s,a)]
\]

Plain English:

```text
old opinion of action
        ↓
experience action
        ↓
see immediate reward + promisingness of next state
        ↓
move opinion a little toward that evidence
```


In [ ]:
# This is HEADLESS training.
# No rendering. Thousands of complete boss attempts run quickly.

def factory(rep="full"):
    return ElderMistReplicaEnv(representation=rep, reward_mode="shaped")

q, rows = train_q(lambda: factory("full"), episodes=12000, seed=42)
train_df = pd.DataFrame(rows)

print(train_df.tail())
print()
print("Last 500 episodes:")
print("win rate:", train_df.tail(500)["win"].mean())
print("mean turns:", train_df.tail(500)["turns"].mean())
print("mean counter slashes:", train_df.tail(500)["counter_slashes"].mean())


In [ ]:
rolling = train_df["return"].rolling(150, min_periods=1).mean()

plt.figure(figsize=(9,4))
plt.plot(train_df["episode"], rolling)
plt.xlabel("Training episode")
plt.ylabel("150-episode rolling return")
plt.title("Q-learning: does the policy improve with experience?")
plt.show()


In [ ]:
rolling_win = train_df["win"].rolling(200, min_periods=1).mean()

plt.figure(figsize=(9,4))
plt.plot(train_df["episode"], rolling_win)
plt.ylim(0, 1)
plt.xlabel("Training episode")
plt.ylabel("200-episode rolling win rate")
plt.title("Win rate during training")
plt.show()


## 6 — Watch the learned policy fight

Now freeze the Q-table and make it greedy:

```text
current compact state
        ↓
look up Q-values
        ↓
choose highest-valued legal action
```

No exploration.


In [ ]:
def learned_policy(env, obs=None):
    return q_policy(q, env)

learned_end, learned_return, learned_df = run_episode(learned_policy, seed=123)

print("win =", learned_end.state.boss_hp <= 0)
print("turns =", learned_end.state.turn)
print("counter slashes =", learned_end.state.counter_slashes)
print("specials cancelled =", learned_end.state.specials_cancelled)
print("return =", round(learned_return, 3))

display(learned_df[[
    "turn","action","reward","boss_hp","sword_active","sword_hp",
    "zale_hp","valere_hp","event"
]].head(30))


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(learned_df["turn"], learned_df["boss_hp"], label="Elder Mist HP")
plt.plot(learned_df["turn"], learned_df["sword_hp"], label="Sword HP")
plt.xlabel("Player action / turn")
plt.ylabel("HP")
plt.title("One learned-policy fight")
plt.legend()
plt.show()


## Why did the learned policy choose that move?

For Q-learning, we can answer this directly: inspect every legal action's current **Q-value** for the represented state.

In [ ]:
from eldermist.qlearning import explain_q_choice

inspect_env = ElderMistReplicaEnv()
inspect_env.reset(seed=123)

explanation = explain_q_choice(q, inspect_env)
print("Represented state:", explanation["represented_state"])
print("Chosen:", explanation["chosen_action"])
print()
display(pd.DataFrame(explanation["ranked_actions"]))


The Q-policy does **not** contain an English rule like “Sword active → Sunball.” It chooses the legal action with the highest learned estimate of future return in this represented state.

## 7 — THIS is the state-representation experiment

Same environment.  
Same Q-learning algorithm.  
Same rewards.

Only change:

> what information counts as the policy's state?

We'll compare:

```text
FULL
actor + HP + boss HP + Sword + Sword HP + charged move + timer + locks + MP + shield

NO_SWORD
everything except Sword state

NO_LOCKS
hide charged-move / lock information

MINIMAL
actor + party HP + boss HP
```

This is the tiny version of Fatimah's question.


In [ ]:
representations = ["full", "no_sword", "no_locks", "minimal"]
comparison = []
trained = {}

for rep in representations:
    q_rep, rows_rep = train_q(
        lambda rep=rep: ElderMistReplicaEnv(
            representation=rep,
            reward_mode="shaped"
        ),
        episodes=8000,
        seed=100,
    )
    trained[rep] = q_rep

    # Fresh held-out seeds, greedy evaluation
    eval_rows = []
    for seed in range(10_000, 10_250):
        e = ElderMistReplicaEnv(representation=rep, reward_mode="shaped")
        obs, _ = e.reset(seed=seed)
        while True:
            a = q_policy(q_rep, e)
            obs, r, term, trunc, _ = e.step(a)
            if term or trunc:
                break

        eval_rows.append({
            "win": int(e.state.boss_hp <= 0),
            "turns": e.state.turn,
            "counter_slashes": e.state.counter_slashes,
            "specials_cancelled": e.state.specials_cancelled,
        })

    edf = pd.DataFrame(eval_rows)
    comparison.append({
        "representation": rep,
        "win_rate": edf["win"].mean(),
        "avg_turns": edf["turns"].mean(),
        "avg_counter_slashes": edf["counter_slashes"].mean(),
        "avg_specials_cancelled": edf["specials_cancelled"].mean(),
    })

comparison_df = pd.DataFrame(comparison)
comparison_df


In [ ]:
plt.figure(figsize=(8,4))
plt.bar(comparison_df["representation"], comparison_df["win_rate"])
plt.ylim(0, 1)
plt.ylabel("Held-out win rate")
plt.title("Which state representation lets the policy act well?")
plt.show()


### What to look for

If hiding Sword information causes:

- more Counter Slashes
- lower win rate
- longer fights

then the experiment says:

> Sword state contains decision-relevant information.

That's more precise than “capture more telemetry.”

A good representation preserves information necessary for choosing actions and predicting relevant consequences.


## 8 — Observation vs representation, one last time

Our simulator can expose **everything**, but the policy does not have to receive everything.

```text
TRUE WORLD STATE
3995 HP
Sword HP
Sword disabled timer
exact lock symbols
MP
combo
shield state
boss turn count
...

         ↓ representation function

STATE REPRESENTATION
(actor, HP buckets, Sword active?, locks, MP buckets, ...)
         ↓
       policy
         ↓
       action
```

For LifeOS / computer use:

```text
TRUE WORLD
      ↓
screenshots + accessibility tree + history + telemetry
      ↓
STATE REPRESENTATION
      ↓
employee/Carl/agent policy
      ↓
next action
```

That is the bridge you've been trying to articulate.


## 9 — Next experiment after this clicks

There are two fun directions:

**A. Partial observability / belief state**

Hide the Sword state from the current observation, but let the agent remember history:

```text
I attacked it twice...
then it disappeared...
it's been two boss turns...
→ I believe the Sword is probably returning soon
```

Now history matters.

**B. Neural policy**

Replace the Q-table with a small MLP:

```text
numeric representation
      ↓
neural-network weights
      ↓
action probabilities
```

and train the weights from reward.

Do A first if your immediate goal is understanding POMDP/state representation.  
Do B first if your immediate goal is understanding how OpenAI Five-like learned policies differ from prompts/LLMs.
